In [ ]:
import torchvision 
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import matplotlib.pyplot as plt 
import torch.optim as optim
import numpy as np
import pandas as pd
import os

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

In [ ]:
# Первая сверточная неиронка 
import torchvision 
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import matplotlib.pyplot as plt 
import torch.optim as optim

BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  
])

train_dataset = torchvision.datasets.MNIST(
    root = "./data",
    train = True,
    download= True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,          
    download=True,
    transform=transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(in_channels=32, out_channels= 64, kernel_size= 3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)
        
        
        
        

    def forward(self, x):
         x = self.pool1(self.relu1(self.conv1(x)))
         x = self.pool2(self.relu2(self.conv2(x)))
         x = self.flatten(x)
         x = self.relu3(self.fc1(x))
         x = self.fc2(x)
         return x 
        
        
        
model = SimpleCNN().to(device) 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) 

def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0 
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        
        loss =criterion(outputs, labels)
        
        loss.backward()
        
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    avg_loss= running_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def test_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0 
    total = 0 
    
    with torch.no_grad(): 
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs =model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    avg_loss =running_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("Погнали учить..... :)")
for epoch in range(EPOCHS):
    train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test_model(model, test_loader, criterion, device)
    
    print(f"ЭПОХАА [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%  | "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
def visualize_predictions(model, test_loader, device, num_images=10):
    model.eval()
    images, labels = next(iter(test_loader))
    images, labels = images.to(device), labels.to(device)
    
    with torch.no_grad():
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
    images = images.cpu() * 0.3081 + 0.1307
    
    fig, axes = plt.subplots(2, 5, figsize=(12,6))
    for i, ax in enumerate(axes.flat):
        if i < num_images:
            ax.imshow(images[i].squeeze(), cmap="gray") 
            ax.set_title(f"Pred: {predicted[i].item()}\nTrue: {labels[i].item()}")
            ax.axis("off")
            
            
        
    
    plt.tight_layout()
    plt.show()           
    
visualize_predictions(model, test_loader, device, num_images=10)    
    
                
    
        
        





In [ ]:
class XY:
    def __init__(self, N, T, seed = None, filename=None):
        self.N = N
        self.T = T
        self.rng = np.random.default_rng(seed)  # Инициализация генератора случайных чисел с фиксированным seed для воспроизводимости
        self.grids = self.load_config(filename)
        self.id_grid = 0
        self.beta = 1.0 / self.T
        self.J = 1.0  # Константа взаимодействия
        self.idx = np.arange(N * N).reshape(N, N)  # Индексы для построения графа
        if self.grids == []:
            self.create_grid(N)
        else:    
            self.grid = self.grids[self.id_grid]
        
    rng = np.random.default_rng()

    def create_grid(self, N):
        # N x N случайных углов в градусах, [0, 360)
         self.grid = self.rng.uniform(0, 360, N**2).reshape(N, N)

    def save_config(self, filename=None, type="npz"):
        """Сохраняет текущую конфигурацию углов в файл."""
        if filename is None:
            if type == "npz":
                filename = f"xy_config_{self.N}x{self.N}-{self.T}.npz"
            else:
                filename = f"xy_config_{self.N}x{self.N}-{self.T}.txt"

        existing = os.path.exists(filename)
        if type == "txt":
            if existing:
                with open(filename, "a") as f:
                    np.savetxt(f, self.grid , fmt="%.3f")
            else:
                    np.savetxt(filename, self.grid, fmt="%.3f")
        else:
            np.savez(filename, self.grids)

    def load_config(self, filename, type="npz"):
        """Загружает конфигурацию углов из файла."""
        if filename is None:
            return []
        
        if type == "npz":
            self.grids = np.load(filename)['arr_0']
        else:
            self.grids = np.loadtxt(filename)
            self.N = int(filename.split('-')[0].split('_')[-1].split('x')[0])  # Извлекаем размер сетки из имени файла
            if type == "txt":
                self.T = float(filename.split('-')[-1].replace('.txt', ''))  # Извлекаем температуру из имени файла
            else:
                self.T = float(filename.split('-')[-1].replace('.npz', ''))  # Извлекаем температуру из имени файла
            
            if self.grids.size % (self.N * self.N) != 0:
                raise ValueError(f"Размер {self.grids.size} не делится на {self.N}x{self.N}")

            self.grids = list(self.grids.reshape(-1, self.N, self.N))  # Список конфигураций  

            return self.grids
    

    def next_config(self):
        """Переходит к следующей конфигурации в списке self.grids."""
        if not self.grids:
            raise ValueError("Нет загруженных конфигураций для перехода.")

        if self.id_grid >= len(self.grids):
            raise IndexError("Индекс конфигурации выходит за пределы списка.")
        
        self.id_grid = (self.id_grid + 1) % len(self.grids)
        self.grid = self.grids[self.id_grid]
        return self.grid

    def hamiltonian(self):
        """Вычисляет гамильтониан модели XY на сетке self.grid."""
        # Преобразуем углы в радианы
        theta = np.deg2rad(self.grid)

        # Вычисляем взаимодействия между соседними спинами
        H = 0.0
        for i in range(self.N):
            for j in range(self.N):
                # Соседи по периодическим границам
                right = (i, (j + 1) % self.N)
                down = ((i + 1) % self.N, j)

                # Вклад от взаимодействия с правым соседом
                H -= np.cos(theta[i, j] - theta[right])
                # Вклад от взаимодействия с нижним соседом
                H -= np.cos(theta[i, j] - theta[down])

        return H

    def swendsen_wang_step(self):
        N, rng = self.N, self.rng
        theta = self.grid
 
        # 0. random axis r = (cos phi, sin phi); projections sigma_i . r
        phi = rng.uniform(0, 2 * np.pi)
        proj = np.cos(theta - phi)
 
        # 1. bonds on right and down neighbours
        rows, cols = [], []
        for axis in (0, 1):
            prod = proj * np.roll(proj, -1, axis=axis)
            same_sign = prod > 0
            p = 1.0 - np.exp(-2.0 * self.beta * self.J * np.where(same_sign, prod, 0.0))
            bond = same_sign & (rng.random((N, N)) < p)
            rows.append(self.idx[bond])
            cols.append(np.roll(self.idx, -1, axis=axis)[bond])
        rows, cols = np.concatenate(rows), np.concatenate(cols)
 
        # 2. clusters (connected components)
        L = N * N
        graph = coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(L, L))
        ncomp, labels = connected_components(graph, directed=False)
 
        # 3. flip each cluster with probability 1/2: reflection sigma -> sigma - 2(sigma.r)r
        flip = rng.random(ncomp) < 0.5
        mask = flip[labels].reshape(N, N)
        theta[mask] = 2 * phi + np.pi - theta[mask]
        self.theta = np.mod(theta, 2 * np.pi)
        return ncomp
     
    def magnetization(self):
        t = self.theta
        return np.hypot(np.cos(t).mean(), np.sin(t).mean())

    def plot(self, ax=None, cmap="hsv"):
        """Рисует стрелку в каждом узле сетки по углу из self.grid."""
        if ax is None:
            _, ax = plt.subplots(figsize=(8, 8))

        theta = np.deg2rad(self.grid)          # градусы -> радианы
        U, V = np.cos(theta), np.sin(theta)    # единичные векторы

        # grid[i, j]: i — строка (ось y), j — столбец (ось x)
        X, Y = np.meshgrid(np.arange(self.N), np.arange(self.N))

        q = ax.quiver(
            X, Y, U, V,
            self.grid,                 # цвет по углу
            cmap=cmap,
            clim=(0, 360),
            angles="xy",
            pivot="mid",
            scale_units="xy",
            scale=1.2,                 # больше -> короче стрелки
            width=0.005,
        )
        cbar = plt.colorbar(q, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Угол, °")

        ax.set_aspect("equal")
        ax.set_xlim(-1, self.N)
        ax.set_ylim(-1, self.N)
        ax.set_title(f"XY-сетка {self.N}×{self.N} при T={self.T}")
        return ax

model = XY(N=10, T=1.0)
model.load_config(filename="xy_config_10x10-1.0.npz")

In [ ]:
#Вторая неиронка под изинга XY
import glob, os, re
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
TC = 0.893


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class XYDataset(torch.utils.data.Dataset):
    def __init__(self, theta, T, tc=TC, augment=False):
        self.x = torch.from_numpy(
            np.stack([np.cos(theta), np.sin(theta)], axis=1)
        ).float()
        self.T = torch.from_numpy(T).float()
        self.y =(self.T > tc).long()
        self.L = self.x.shape[-1]
        self.augment = augment
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, i):
        x = self.x[i]
        if self.augment:
            a = random.random() * 2 * math.pi
            c, s = math.cos(a), math.sin(a)
            x = torch.stack([c * x[0] - s * x[1],
                             s * x[0] + c * x[1]])
            x =torch.roll(x, shifts=(random.randrange(self.L), random.randrange(self.L)),
                          dims=(1, 2))           
        return x, self.y[i]
    

CONFIG_DIR = ""     
N_SIZE = 10  
       
raw_list, T_list = [], []
for path in sorted(glob.glob(os.path.join(CONFIG_DIR, f"xy_config_{N_SIZE}x{N_SIZE}-*.npz"))):
    t = float(re.search(r"-(\d+(?:\.\d+)?)\.npz$", os.path.basename(path)).group(1))
    arr = np.load(path)["arr_0"].reshape(-1, N_SIZE, N_SIZE)   # (n, N, N), градусы
    raw_list.append(arr)
    T_list.append(np.full(len(arr), t, dtype=np.float32))

if not raw_list:
    raise FileNotFoundError(f"Нет файлов xy_config_{N_SIZE}x{N_SIZE}-*.npz в {os.path.abspath(CONFIG_DIR)}")

raw = np.concatenate(raw_list)
T = np.concatenate(T_list)
def to_theta(raw):
    if raw.ndim == 3:
        theta = raw
        if theta.max() > 2 * np.pi + 1e-3:
            theta = np.deg2rad(theta)
        return theta.astype(np.float32)
    if raw.ndim == 4 and raw.shape[-1] == 2:
        return np.arctan2(raw[..., 1], raw[..., 0]).astype(np.float32)
    if raw.ndim == 4 and raw.shape[1] == 2:
        return np.arctan2(raw[:, 1], raw[:, 0]).astype(np.float32)
    raise ValueError(f"Неожиданная форма данных: {raw.shape}")

theta = to_theta(raw)
L = theta.shape[-1]
print(len(T) , theta.shape[1:], T.min(), T.max())


rng = np.random.default_rng(1)
idx = rng.permutation(len(T))
n_test = len(T) // 5
test_idx = idx[:n_test]
train_idx = idx[n_test:]
train_idx = train_idx[np.abs(T[train_idx] - TC) > 0.2]

train_dataset = XYDataset(theta[train_idx], T[train_idx], augment=True)
test_dataset = XYDataset(theta[test_idx], T[test_idx])

print("Train:", len(train_dataset), "Test:", len(test_dataset))
print("train:класс", (train_dataset.y == 1).sum().item())

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

class SimpleCNN(nn.Module):
    def __init__(self, L, num_classes=2):
        super(SimpleCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=2, out_channels=32, kernel_size=3,
                              padding=1, padding_mode="circular")
        
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3,
                               padding=1, padding_mode="circular")
        
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * (L // 4) * (L // 4), 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x
    
model = SimpleCNN(L=L).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    avg_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def test_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    avg_loss = running_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

print("Погнали учить..... :)")
for epoch in range(EPOCHS):
    train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test_model(model, test_loader, criterion, device)
    
    print(f"ЭПОХАА [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "                  
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
def plot_phase_curve(model, loader, dataset, device):
    model.eval()
    probs = []
    with torch.no_grad():
        for images, _ in loader:
            out = model(images.to(device))
            probs.append(torch.softmax(out, dim=1)[:, 1].cpu())
    probs = torch.cat(probs).numpy()
    Ts = dataset.T.numpy()
    
    uniq = np.unique(Ts)
    mean_p = [probs[Ts == t].mean() for t in uniq]
    
    plt.plot(uniq, mean_p, "o-")
    plt.axvline(TC, color="r", ls="--", label="T_BKT ≈ 0.893")
    plt.xlabel("T")
    plt.ylabel("P(неупорядоченная фаза)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show() 
           
plot_phase_curve(model, test_loader, test_dataset, device)       

In [ ]:
# Нейронка для модели XY: классификация фаз + график выходного слоя

import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
TC = 0.893
EXCLUDE_WINDOW = 0.2   


DATASETS = [("xy_config_20x20-1.0.npz", None)]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



class XYDataset(torch.utils.data.Dataset):
    def __init__(self, theta, T, tc=TC, augment=False):
        self.x = torch.from_numpy(
            np.stack([np.cos(theta), np.sin(theta)], axis=1)
        ).float()
        self.T = torch.from_numpy(T).float()
        self.y = (self.T > tc).long()
        self.L = self.x.shape[-1]
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.x[i]
        if self.augment:
            a = random.random() * 2 * math.pi
            c, s = math.cos(a), math.sin(a)
            x = torch.stack([c * x[0] - s * x[1],
                             s * x[0] + c * x[1]])
            x = torch.roll(x, shifts=(random.randrange(self.L), random.randrange(self.L)),
                           dims=(1, 2))
        return x, self.y[i]


def to_theta(raw):
    if raw.ndim == 3:
        theta = raw
        if theta.max() > 2 * np.pi + 1e-3:
            theta = np.deg2rad(theta)
        return theta.astype(np.float32)
    if raw.ndim == 4 and raw.shape[-1] == 2:
        return np.arctan2(raw[..., 1], raw[..., 0]).astype(np.float32)
    if raw.ndim == 4 and raw.shape[1] == 2:
        return np.arctan2(raw[:, 1], raw[:, 0]).astype(np.float32)
    raise ValueError(f"Неожиданная форма данных: {raw.shape}")


def load_dataset(angles_path, temps_path=None):
    
    if angles_path.endswith(".npz"):
        d = np.load(angles_path)
        raw = d["theta"] if "theta" in d else d[d.files[0]]
        T = d["T"] if (temps_path is None and "T" in d) else None
    elif angles_path.endswith(".npy"):
        raw, T = np.load(angles_path), None
    else:  # txt: конфигурации друг под другом
        flat = np.loadtxt(angles_path)
        raw, T = flat.reshape(-1, flat.shape[1], flat.shape[1]), None

    if temps_path is not None:
        if temps_path.endswith(".npy"):
            T = np.load(temps_path)
        elif temps_path.endswith(".npz"):
            d = np.load(temps_path)
            T = d["T"] if "T" in d else d[d.files[0]]
        else:
            T = np.loadtxt(temps_path, delimiter="," if temps_path.endswith(".csv") else None)
    if T is None:
        raise ValueError("Не найдены температуры: укажите файл с температурами")

    T = np.asarray(T, dtype=np.float32).ravel()
    theta = to_theta(raw)
    assert len(T) == len(theta), f"Углов {len(theta)}, температур {len(T)}: не совпадает"
    return theta, T



class SimpleCNN(nn.Module):
    def __init__(self, L, num_classes=2):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=2, out_channels=32, kernel_size=3,
                               padding=1, padding_mode="circular")
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3,
                               padding=1, padding_mode="circular")
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * (L // 4) * (L // 4), 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x


def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def test_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy



def get_outputs(model, dataset, device):
   
    loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False)
    model.eval()
    probs = []
    with torch.no_grad():
        for x, _ in loader:
            probs.append(torch.softmax(model(x.to(device)), dim=1).cpu())
    probs = torch.cat(probs).numpy()          # (n, 2)
    Ts = dataset.T.numpy()
    uniq = np.unique(Ts)
    m0 = np.array([probs[Ts == t, 0].mean() for t in uniq])
    s0 = np.array([probs[Ts == t, 0].std() for t in uniq])
    m1 = np.array([probs[Ts == t, 1].mean() for t in uniq])
    s1 = np.array([probs[Ts == t, 1].std() for t in uniq])
    return uniq, m0, s0, m1, s1


def plot_output_layer(results, tc=TC, save="output_layer.png"):
    """results: {L: (T, mean_ordered, std_ordered, mean_disordered, std_disordered)}"""
    plt.rcParams["font.family"] = "serif"
    sizes = sorted(results)
    blues = plt.cm.Blues(np.linspace(0.4, 0.9, len(sizes)))
    reds = plt.cm.Reds(np.linspace(0.4, 0.9, len(sizes)))
    markers = ["o", "s", "D", "^", "v"]

    fig, ax = plt.subplots(figsize=(7, 5))
    for k, L in enumerate(sizes):
        T, m0, s0, m1, s1 = results[L]
        ax.errorbar(T, m0, yerr=s0, color=blues[k], marker=markers[k % len(markers)],
                    ms=4, ls="--", lw=1.3, capsize=2, label=f"{L}×{L}")
        ax.errorbar(T, m1, yerr=s1, color=reds[k], marker=markers[k % len(markers)],
                    ms=4, ls="-", lw=1.3, capsize=2)

    ax.axvline(tc, color="k", ls="--", lw=1)
    ax.set_xlabel(r"$T/J$", fontsize=13)
    ax.set_ylabel("Выходной слой нейронной сети", fontsize=13)
    ax.set_ylim(-0.02, 1.05)
    ax.grid(alpha=0.3, ls=":")
    ax.legend(loc="center right", fontsize=10)
    fig.tight_layout()
    fig.savefig(save, dpi=200)
    plt.show()



def run_one(angles_path, temps_path):
    theta, T = load_dataset(angles_path, temps_path)
    L = theta.shape[-1]
    print(f"\n=== L = {L} ===")
    print(len(T), theta.shape[1:], T.min(), T.max())

    rng = np.random.default_rng(1)
    idx = rng.permutation(len(T))
    n_test = len(T) // 5
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    if EXCLUDE_WINDOW > 0:
        train_idx = train_idx[np.abs(T[train_idx] - TC) > EXCLUDE_WINDOW]

    train_dataset = XYDataset(theta[train_idx], T[train_idx], augment=True)
    test_dataset = XYDataset(theta[test_idx], T[test_idx])

    print("Train:", len(train_dataset), "Test:", len(test_dataset))
    print("train:класс", (train_dataset.y == 1).sum().item())

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = SimpleCNN(L=L).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print("Погнали учить..... :)")
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = test_model(model, test_loader, criterion, device)
        print(f"ЭПОХАА [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

    return L, get_outputs(model, test_dataset, device)



results = {}
for angles_path, temps_path in DATASETS:
    L, out = run_one(angles_path, temps_path)
    results[L] = out

plot_output_layer(results)

In [ ]:
d = np.load("xy_config_10x10-1.0.npz")
print(d.files)
for k in d.files:
    print(k, d[k].shape, d[k].dtype)

In [ ]:
print(np.load("xy_config_10x10-1.0.npz")["arr_0"].shape)

In [ ]:

import glob, os, re
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
TC = 0.893
WEIGHTS_PATH = "xy_cnn_weights.pth"
TRAIN = True
CONFIG_DIR = ""     
N_SIZE = 20         

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



class XYDataset(torch.utils.data.Dataset):
    def __init__(self, theta, T, tc=TC, augment=False):
       
        self.x = torch.from_numpy(
            np.stack([np.cos(theta), np.sin(theta)], axis=1)
        ).float()
        self.T = torch.from_numpy(T).float()
        self.y = (self.T > tc).long()     
        self.L = self.x.shape[-1]
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.x[i]
        if self.augment:
           
            a = random.random() * 2 * math.pi
            c, s = math.cos(a), math.sin(a)
            x = torch.stack([c * x[0] - s * x[1],
                             s * x[0] + c * x[1]])
           
            x = torch.roll(x, shifts=(random.randrange(self.L), random.randrange(self.L)),
                           dims=(1, 2))
        return x, self.y[i]



raw_list, T_list = [], []
for path in sorted(glob.glob(os.path.join(CONFIG_DIR, f"xy_config_{N_SIZE}x{N_SIZE}-*.npz"))):
    t = float(re.search(r"-(\d+(?:\.\d+)?)\.npz$", os.path.basename(path)).group(1))
    arr = np.load(path)["arr_0"].reshape(-1, N_SIZE, N_SIZE)   # (n, N, N), градусы
    raw_list.append(arr)
    T_list.append(np.full(len(arr), t, dtype=np.float32))

if not raw_list:
    raise FileNotFoundError(f"Нет файлов xy_config_{N_SIZE}x{N_SIZE}-*.npz в {os.path.abspath(CONFIG_DIR)}")

raw = np.concatenate(raw_list)
T = np.concatenate(T_list)


def to_theta(raw):
    if raw.ndim == 3:
        theta = raw
        if theta.max() > 2 * np.pi + 1e-3:    
            theta = np.deg2rad(theta)
        return theta.astype(np.float32)
    if raw.ndim == 4 and raw.shape[-1] == 2:
        return np.arctan2(raw[..., 1], raw[..., 0]).astype(np.float32)
    if raw.ndim == 4 and raw.shape[1] == 2:
        return np.arctan2(raw[:, 1], raw[:, 0]).astype(np.float32)
    raise ValueError(f"Неожиданная форма данных: {raw.shape}")


theta = to_theta(raw)
L = theta.shape[-1]
print(len(T), theta.shape[1:], T.min(), T.max())


rng = np.random.default_rng(1)
idx = rng.permutation(len(T))
n_test = len(T) // 5
test_idx = idx[:n_test]
train_idx = idx[n_test:]
# в обучении убираем область около T_c (|T - TC| <= 0.2), сеть её не видит
train_idx = train_idx[np.abs(T[train_idx] - TC) > 0.2]

train_dataset = XYDataset(theta[train_idx], T[train_idx], augment=True)
test_dataset = XYDataset(theta[test_idx], T[test_idx])

print("Train:", len(train_dataset), "Test:", len(test_dataset))
print("train: класс 1 (неупорядоченная):", (train_dataset.y == 1).sum().item())

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False   # shuffle=False важен для графиков
)



class SimpleCNN(nn.Module):
    def __init__(self, L, num_classes=2):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=2, out_channels=32, kernel_size=3,
                               padding=1, padding_mode="circular")
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3,
                               padding=1, padding_mode="circular")
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * (L // 4) * (L // 4), 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x


model = SimpleCNN(L=L).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)



def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def test_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


if TRAIN:
    print("Погнали учить..... :)")
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = test_model(model, test_loader, criterion, device)

        print(f"ЭПОХАА [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
              f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

    torch.save(model.state_dict(), WEIGHTS_PATH)      # сохранить после обучения
    print("Веса сохранены в", WEIGHTS_PATH)
else:
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    model.eval()
    print("Веса загружены из", WEIGHTS_PATH)


def xy_energy(theta):
    
    theta = theta.astype(np.float64)       
    right = np.roll(theta, 1, axis=2)      
    down = np.roll(theta, 1, axis=1)       
    return -(np.cos(theta - right) + np.cos(theta - down)).sum(axis=(1, 2))


def plot_heat_capacity(theta, T):
    N = theta.shape[-1] ** 2
    E = xy_energy(theta)
    uniq = np.unique(T)

    # C = (<E^2> - <E>^2) / (N T^2), k_B = 1
    C = np.array([E[T == t].var() / (N * float(t) ** 2) for t in uniq])
    t_peak = uniq[np.argmax(C)]

    plt.figure()
    plt.plot(uniq, C, "o-", color="firebrick", ms=4)
    plt.axvline(t_peak, color="k", ls="--", lw=1, label=f"пик C(T): T = {t_peak:.3f}")
    plt.axvline(TC, color="gray", ls=":", lw=1, label=f"T_BKT ≈ {TC}")
    plt.xlabel("T / J")
    plt.ylabel("C(T)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig("heat_capacity.png", dpi=150)



def plot_output_layer(model, loader, dataset, device):
    model.eval()
    probs = []
    with torch.no_grad():
        for images, _ in loader:
            out = model(images.to(device))
            probs.append(torch.softmax(out, dim=1).cpu())    
    probs = torch.cat(probs).numpy()
    Ts = dataset.T.numpy()

    uniq = np.unique(Ts)
    p_ord = np.array([probs[Ts == t, 0].mean() for t in uniq])   
    p_dis = np.array([probs[Ts == t, 1].mean() for t in uniq])   

   
    t_cross = None
    sign_change = np.where(np.diff(np.sign(p_dis - 0.5)))[0]
    if len(sign_change) > 0:
        i = sign_change[0]
        t_cross = uniq[i] + (0.5 - p_dis[i]) * (uniq[i + 1] - uniq[i]) / (p_dis[i + 1] - p_dis[i])

    plt.figure()
    plt.plot(uniq, p_ord, "s--", color="royalblue", ms=4, label="упорядоченная фаза")
    plt.plot(uniq, p_dis, "o-", color="firebrick", ms=4, label="неупорядоченная фаза")
    plt.axhline(0.5, color="gray", lw=0.5)
    if t_cross is not None:
        plt.axvline(t_cross, color="k", ls="--", lw=1, label=f"пересечение: T = {t_cross:.3f}")
    plt.axvline(TC, color="gray", ls=":", lw=1, label=f"T_BKT ≈ {TC}")
    plt.xlabel("T / J")
    plt.ylabel("Выходной слой нейронной сети")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig("output_layer.png", dpi=150)


plot_heat_capacity(theta, T)
plot_output_layer(model, test_loader, test_dataset, device)
plt.show()   

MemoryError: Unable to allocate 1.49 GiB for an array with shape (1001000, 20, 20) and data type float32

In [6]:
import struct, os, psutil
print("разрядность Python:", struct.calcsize("P") * 8)          # должно быть 64
print("занято процессом, GiB:", psutil.Process(os.getpid()).memory_info().rss / 2**30)
print("свободно в системе, GiB:", psutil.virtual_memory().available / 2**30)

разрядность Python: 64
занято процессом, GiB: 0.2608833312988281
свободно в системе, GiB: 0.4742317199707031


In [3]:

import os
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components


class XY:
    def __init__(self, N, T, seed=None):
        self.N = N
        self.J = 1.0                                   
        self.rng = np.random.default_rng(seed)        
        self.idx = np.arange(N * N).reshape(N, N)     
        self.grids = []                                
        self.energies = []                             
        self.id_grid = 0
        self.set_temperature(T)
        self.create_grid(N)

    def set_temperature(self, T):
        self.T = T
        self.beta = 1.0 / T

    def create_grid(self, N):
     
        self.grid = self.rng.uniform(0, 360, N ** 2).reshape(N, N)

 
    def save_config(self, filename=None, type="npz"):
        """npz: сохраняет ВСЕ накопленные конфигурации self.grids (ключ arr_0, градусы).
        txt: дописывает текущую конфигурацию self.grid."""
        if filename is None:
            ext = "npz" if type == "npz" else "txt"
            filename = f"xy_config_{self.N}x{self.N}-{self.T}.{ext}"

        if type == "txt":
            if os.path.exists(filename):
                with open(filename, "a") as f:
                    np.savetxt(f, self.grid, fmt="%.3f")
            else:
                np.savetxt(filename, self.grid, fmt="%.3f")
        else:
            
            np.savez(filename, np.asarray(self.grids, dtype=np.float32))

    def load_config(self, filename, type="npz"):
        """Загружает конфигурации; N и T берутся из имени файла."""
        m = re.search(r"xy_config_(\d+)x\d+-(\d+(?:\.\d+)?)\.(?:npz|txt)$",
                      os.path.basename(filename))
        if m is None:
            raise ValueError(f"Не могу разобрать имя файла: {filename}")
        self.N = int(m.group(1))
        self.set_temperature(float(m.group(2)))
        self.idx = np.arange(self.N ** 2).reshape(self.N, self.N)

        if type == "npz":
            data = np.load(filename)["arr_0"]
        else:
            data = np.loadtxt(filename)

        if data.size % (self.N * self.N) != 0:
            raise ValueError(f"Размер {data.size} не делится на {self.N}x{self.N}")

        self.grids = data.reshape(-1, self.N, self.N)
        self.id_grid = 0
        self.grid = self.grids[0].astype(float)      
        return self.grids

    def next_config(self):
        """Переходит к следующей конфигурации в self.grids."""
        if len(self.grids) == 0:
            raise ValueError("Нет загруженных конфигураций для перехода.")
        self.id_grid = (self.id_grid + 1) % len(self.grids)
        self.grid = self.grids[self.id_grid].astype(float)
        return self.grid


    def hamiltonian(self):
        """E = -J * sum cos(theta_i - theta_j) по соседям (без циклов, периодические границы)."""
        theta = np.deg2rad(self.grid)
        right = np.roll(theta, -1, axis=1)
        down = np.roll(theta, -1, axis=0)
        return -self.J * (np.cos(theta - right) + np.cos(theta - down)).sum()

    def swendsen_wang_step(self):
        N, rng = self.N, self.rng
        theta = np.deg2rad(self.grid)       

        
        phi = rng.uniform(0, 2 * np.pi)
        proj = np.cos(theta - phi)

       
        rows, cols = [], []
        for axis in (0, 1):
            prod = proj * np.roll(proj, -1, axis=axis)
            same_sign = prod > 0
            p = 1.0 - np.exp(-2.0 * self.beta * self.J * np.where(same_sign, prod, 0.0))
            bond = same_sign & (rng.random((N, N)) < p)
            rows.append(self.idx[bond])
            cols.append(np.roll(self.idx, -1, axis=axis)[bond])
        rows, cols = np.concatenate(rows), np.concatenate(cols)

    
        L = N * N
        graph = coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(L, L))
        ncomp, labels = connected_components(graph, directed=False)

       
        flip = rng.random(ncomp) < 0.5
        mask = flip[labels].reshape(N, N)
        theta[mask] = 2 * phi + np.pi - theta[mask]

        self.grid = np.rad2deg(np.mod(theta, 2 * np.pi))
        return ncomp

    def magnetization(self):
        t = np.deg2rad(self.grid)
        return np.hypot(np.cos(t).mean(), np.sin(t).mean())

   
    def thermalize(self, n_steps=500):
        """Прогрев: отбрасываем начальные шаги, пока система не придёт к равновесию."""
        for _ in range(n_steps):
            self.swendsen_wang_step()

    def generate(self, n_samples=1000, n_therm=500, n_skip=3):
        """Накапливает n_samples конфигураций; между снимками n_skip шагов SW,
        чтобы соседние конфигурации были почти независимы."""
        self.thermalize(n_therm)
        out = np.empty((n_samples, self.N, self.N), dtype=np.float32)
        energies = np.empty(n_samples)
        for k in range(n_samples):
            for _ in range(n_skip):
                self.swendsen_wang_step()
            out[k] = self.grid
            energies[k] = self.hamiltonian()
        self.grids = out
        self.energies = energies
        return out

    def heat_capacity(self):
        """C на один спин: (<E^2> - <E>^2) / (N^2 * T^2), k_B = 1.
        Считается по накопленным энергиям (после generate)."""
        E = np.asarray(self.energies)
        return E.var() / (self.N ** 2 * self.T ** 2)

    def plot(self, ax=None, cmap="hsv"):
        """Рисует стрелку в каждом узле сетки по углу из self.grid."""
        if ax is None:
            _, ax = plt.subplots(figsize=(8, 8))

        theta = np.deg2rad(self.grid)
        U, V = np.cos(theta), np.sin(theta)
        X, Y = np.meshgrid(np.arange(self.N), np.arange(self.N))

        q = ax.quiver(
            X, Y, U, V, self.grid,
            cmap=cmap, clim=(0, 360),
            angles="xy", pivot="mid",
            scale_units="xy", scale=1.2, width=0.005,
        )
        cbar = plt.colorbar(q, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Угол, °")

        ax.set_aspect("equal")
        ax.set_xlim(-1, self.N)
        ax.set_ylim(-1, self.N)
        ax.set_title(f"XY-сетка {self.N}×{self.N} при T={self.T}")
        return ax


def make_dataset(N, temps, n_samples=1000, n_therm=500, n_skip=3, seed=0, out_dir="."):
    """Для каждой температуры генерирует конфигурации и пишет файл
    xy_config_{N}x{N}-{T}.npz (массив (n_samples, N, N), градусы, ключ arr_0)."""
    os.makedirs(out_dir, exist_ok=True)
    for k, T in enumerate(temps):
        sim = XY(N, float(T), seed=seed + k)
        sim.generate(n_samples=n_samples, n_therm=n_therm, n_skip=n_skip)
        filename = os.path.join(out_dir, f"xy_config_{N}x{N}-{T:.3f}.npz")
        sim.save_config(filename=filename)
        print(f"[{k + 1}/{len(temps)}] T={T:.3f}  "
              f"<E>/N^2={sim.energies.mean() / N ** 2:.3f}  "
              f"C={sim.heat_capacity():.3f}  -> {filename}")


if __name__ == "__main__":
    N = 20
    temps = np.round(np.arange(0.50, 1.50 + 1e-9, 0.00025), 3)    # 0.5 ... 1.5, шаг 0.025
    make_dataset(N, temps, n_samples=1000, n_therm=500, n_skip=3, out_dir=".")

[1/4001] T=0.500  <E>/N^2=-1.731  C=0.509  -> .\xy_config_20x20-0.500.npz
[2/4001] T=0.500  <E>/N^2=-1.731  C=0.557  -> .\xy_config_20x20-0.500.npz
[3/4001] T=0.500  <E>/N^2=-1.728  C=0.621  -> .\xy_config_20x20-0.500.npz
[4/4001] T=0.501  <E>/N^2=-1.731  C=0.525  -> .\xy_config_20x20-0.501.npz
[5/4001] T=0.501  <E>/N^2=-1.729  C=0.608  -> .\xy_config_20x20-0.501.npz
[6/4001] T=0.501  <E>/N^2=-1.728  C=0.622  -> .\xy_config_20x20-0.501.npz
[7/4001] T=0.501  <E>/N^2=-1.729  C=0.574  -> .\xy_config_20x20-0.501.npz
[8/4001] T=0.502  <E>/N^2=-1.728  C=0.573  -> .\xy_config_20x20-0.502.npz
[9/4001] T=0.502  <E>/N^2=-1.728  C=0.543  -> .\xy_config_20x20-0.502.npz
[10/4001] T=0.502  <E>/N^2=-1.730  C=0.581  -> .\xy_config_20x20-0.502.npz
[11/4001] T=0.502  <E>/N^2=-1.727  C=0.562  -> .\xy_config_20x20-0.502.npz
[12/4001] T=0.503  <E>/N^2=-1.729  C=0.626  -> .\xy_config_20x20-0.503.npz
[13/4001] T=0.503  <E>/N^2=-1.728  C=0.633  -> .\xy_config_20x20-0.503.npz
[14/4001] T=0.503  <E>/N^2=-1.730 